# 🔐 Global Data Breaches Analysis — HaveIBeenPwned
## Notebook 02 — Nettoyage & Feature Engineering

**Source** : `../data/breaches_raw.csv` (généré dans le Notebook 01)  
**Auteur** : Esmel Amari (Phil)  
**Objectif** : Nettoyer les données brutes, parser les dates, créer de nouvelles variables utiles et exporter un dataset propre prêt pour l'analyse.

---

## 📚 1. Import des librairies

In [1]:
import pandas as pd
import numpy as np
import ast
import os
from datetime import datetime

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)

print("✅ Librairies importées avec succès")
print(f"📅 Date d'exécution : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ Librairies importées avec succès
📅 Date d'exécution : 2026-04-18 05:30:56


## 📂 2. Chargement des données brutes

In [2]:
df = pd.read_csv('../data/breaches_raw.csv')

print(f"📐 Dimensions : {df.shape[0]} lignes x {df.shape[1]} colonnes")
print(f"\n📋 Colonnes disponibles :")
for col in df.columns:
    print(f"  • {col}")

📐 Dimensions : 974 lignes x 20 colonnes

📋 Colonnes disponibles :
  • Name
  • Title
  • Domain
  • BreachDate
  • AddedDate
  • ModifiedDate
  • PwnCount
  • Description
  • LogoPath
  • Attribution
  • DisclosureUrl
  • DataClasses
  • IsVerified
  • IsFabricated
  • IsSensitive
  • IsRetired
  • IsSpamList
  • IsMalware
  • IsSubscriptionFree
  • IsStealerLog


In [3]:
# Sauvegarde d'une copie de travail
df_clean = df.copy()
print("✅ Copie de travail créée : df_clean")

✅ Copie de travail créée : df_clean


## 🔍 3. Audit de qualité des données

In [4]:
# Valeurs manquantes
print("🕳️  Audit des valeurs manquantes :")
print("-" * 50)
missing = df_clean.isnull().sum()
missing_pct = (missing / len(df_clean) * 100).round(2)
audit = pd.DataFrame({
    'Type': df_clean.dtypes,
    'Manquants': missing,
    'Pourcentage (%)': missing_pct
})
print(audit)
print(f"\n📊 Total valeurs manquantes : {missing.sum()}")

🕳️  Audit des valeurs manquantes :
--------------------------------------------------
                      Type  Manquants  Pourcentage (%)
Name                object          0             0.00
Title               object          0             0.00
Domain              object         52             5.34
BreachDate          object          0             0.00
AddedDate           object          0             0.00
ModifiedDate        object          0             0.00
PwnCount             int64          0             0.00
Description         object          0             0.00
LogoPath            object          0             0.00
Attribution         object        856            87.89
DisclosureUrl       object        950            97.54
DataClasses         object          0             0.00
IsVerified            bool          0             0.00
IsFabricated          bool          0             0.00
IsSensitive           bool          0             0.00
IsRetired             bool        

In [5]:
# Doublons
duplicates = df_clean.duplicated().sum()
print(f"🔁 Doublons détectés : {duplicates}")
if duplicates > 0:
    df_clean = df_clean.drop_duplicates()
    print(f"✅ Doublons supprimés. Nouvelles dimensions : {df_clean.shape}")
else:
    print("✅ Aucun doublon trouvé")

🔁 Doublons détectés : 0
✅ Aucun doublon trouvé


## 📅 4. Parsing des colonnes de dates

In [6]:
# Conversion des colonnes de dates
date_cols = ['BreachDate', 'AddedDate', 'ModifiedDate']
date_cols_present = [c for c in date_cols if c in df_clean.columns]

for col in date_cols_present:
    df_clean[col] = pd.to_datetime(df_clean[col], errors='coerce')
    null_count = df_clean[col].isnull().sum()
    print(f"✅ {col:15s} → datetime | Nulls après conversion : {null_count}")

print(f"\n📅 Période couverte :")
print(f"   Première fuite  : {df_clean['BreachDate'].min().strftime('%Y-%m-%d')}")
print(f"   Dernière fuite  : {df_clean['BreachDate'].max().strftime('%Y-%m-%d')}")
print(f"   Span total      : {(df_clean['BreachDate'].max() - df_clean['BreachDate'].min()).days} jours")

✅ BreachDate      → datetime | Nulls après conversion : 0
✅ AddedDate       → datetime | Nulls après conversion : 0
✅ ModifiedDate    → datetime | Nulls après conversion : 0

📅 Période couverte :
   Première fuite  : 2007-07-12
   Dernière fuite  : 2026-04-10
   Span total      : 6847 jours


## ⚙️ 5. Feature Engineering — Nouvelles variables

In [7]:
# ── 5.1 Variables temporelles ──────────────────────────────
df_clean['Year']      = df_clean['BreachDate'].dt.year
df_clean['Month']     = df_clean['BreachDate'].dt.month
df_clean['Quarter']   = df_clean['BreachDate'].dt.quarter
df_clean['YearMonth'] = df_clean['BreachDate'].dt.to_period('M').astype(str)
df_clean['Decade']    = (df_clean['Year'] // 10 * 10).astype(str) + 's'

print("✅ Variables temporelles créées : Year, Month, Quarter, YearMonth, Decade")
print(f"   Années couvertes : {sorted(df_clean['Year'].dropna().unique().tolist()[:5])} ... {sorted(df_clean['Year'].dropna().unique().tolist()[-3:])}")

✅ Variables temporelles créées : Year, Month, Quarter, YearMonth, Decade
   Années couvertes : [2011, 2012, 2015, 2016, 2020] ... [2007, 2009, 2010]


In [8]:
# ── 5.2 Catégorisation de la taille de la fuite ────────────
def categorize_size(n):
    if pd.isna(n):       return 'Inconnu'
    elif n < 100_000:    return 'Petite (<100K)'
    elif n < 1_000_000:  return 'Moyenne (100K–1M)'
    elif n < 10_000_000: return 'Grande (1M–10M)'
    else:                return 'Massive (>10M)'

df_clean['BreachSize'] = df_clean['PwnCount'].apply(categorize_size)

print("✅ Variable BreachSize créée")
print(df_clean['BreachSize'].value_counts().to_string())

✅ Variable BreachSize créée
BreachSize
Moyenne (100K–1M)    339
Grande (1M–10M)      317
Massive (>10M)       173
Petite (<100K)       145


In [9]:
# ── 5.3 Catégorisation du secteur d'activité ───────────────
sector_keywords = {
    'Tech & Réseaux sociaux' : ['google', 'facebook', 'twitter', 'linkedin', 'adobe', 'microsoft',
                                 'apple', 'yahoo', 'myspace', 'tumblr', 'snapchat', 'discord',
                                 'github', 'dropbox', 'lastpass', 'twitch', 'reddit'],
    'E-commerce & Retail'    : ['amazon', 'ebay', 'etsy', 'aliexpress', 'shopify', 'walmart',
                                 'target', 'shop', 'store', 'market'],
    'Finance & Crypto'       : ['paypal', 'bank', 'bitcoin', 'crypto', 'coinbase', 'binance',
                                 'finance', 'capital', 'invest', 'trading'],
    'Gaming'                 : ['game', 'gaming', 'steam', 'xbox', 'playstation', 'nintendo',
                                 'riot', 'epic', 'activision', 'ubisoft', 'zynga'],
    'Santé & Médical'        : ['health', 'medical', 'hospital', 'clinic', 'pharma', 'care',
                                 'med', 'dental', 'therapy'],
    'Éducation'              : ['edu', 'school', 'university', 'college', 'learn', 'course',
                                 'academy', 'student'],
    'Médias & Divertissement': ['news', 'media', 'entertainment', 'music', 'movie', 'tv',
                                 'radio', 'podcast', 'sport', 'netflix'],
    'Voyage & Hôtellerie'    : ['hotel', 'travel', 'airline', 'booking', 'airbnb', 'trip',
                                 'flight', 'tourism'],
    'Télécoms'               : ['telecom', 'mobile', 'phone', 'carrier', 'wireless', 'network'],
}

def get_sector(domain):
    if pd.isna(domain) or domain == '':
        return 'Autre / Inconnu'
    domain_lower = str(domain).lower()
    for sector, keywords in sector_keywords.items():
        if any(kw in domain_lower for kw in keywords):
            return sector
    return 'Autre / Inconnu'

df_clean['Sector'] = df_clean['Domain'].apply(get_sector)

print("✅ Variable Sector créée")
print(f"\n📊 Répartition par secteur :")
print(df_clean['Sector'].value_counts().to_string())

✅ Variable Sector créée

📊 Répartition par secteur :
Sector
Autre / Inconnu            877
Gaming                      34
Médias & Divertissement     15
Tech & Réseaux sociaux      13
E-commerce & Retail         12
Télécoms                     8
Santé & Médical              5
Voyage & Hôtellerie          4
Finance & Crypto             3
Éducation                    3


In [10]:
# ── 5.4 Nombre de types de données exposées ────────────────
def parse_dataclasses(val):
    try:
        if isinstance(val, list):  return val
        if pd.isna(val):           return []
        return ast.literal_eval(val)
    except:
        return []

df_clean['DataClasses_parsed'] = df_clean['DataClasses'].apply(parse_dataclasses)
df_clean['NbDataTypes']        = df_clean['DataClasses_parsed'].apply(len)

print("✅ Variable NbDataTypes créée")
print(f"   Moyenne de types exposés par fuite : {df_clean['NbDataTypes'].mean():.1f}")
print(f"   Maximum de types dans une fuite    : {df_clean['NbDataTypes'].max()}")

✅ Variable NbDataTypes créée
   Moyenne de types exposés par fuite : 5.2
   Maximum de types dans une fuite    : 25


In [11]:
# ── 5.5 Score de gravité ───────────────────────────────────
# Formule : log(PwnCount) x NbDataTypes x (1.5 si Verified) x (2 si Sensitive)
def compute_severity(row):
    try:
        base     = np.log10(max(row['PwnCount'], 1))
        types    = max(row['NbDataTypes'], 1)
        verified = 1.5 if row.get('IsVerified', False) else 1.0
        sensitive = 2.0 if row.get('IsSensitive', False) else 1.0
        return round(base * types * verified * sensitive, 2)
    except:
        return 0.0

df_clean['SeverityScore'] = df_clean.apply(compute_severity, axis=1)

print("✅ Variable SeverityScore créée")
print(f"   Score min   : {df_clean['SeverityScore'].min()}")
print(f"   Score moyen : {df_clean['SeverityScore'].mean():.2f}")
print(f"   Score max   : {df_clean['SeverityScore'].max()}")
print(f"\n🔥 Top 5 fuites par SeverityScore :")
top_sev = df_clean[['Name', 'SeverityScore', 'PwnCount', 'NbDataTypes']].sort_values('SeverityScore', ascending=False).head(5)
print(top_sev.to_string(index=False))

✅ Variable SeverityScore créée
   Score min   : 6.04
   Score moyen : 51.79
   Score max   : 557.82

🔥 Top 5 fuites par SeverityScore :
           Name  SeverityScore  PwnCount  NbDataTypes
          Mate1         557.82  27393015           25
      Zoosk2020         376.32  23927853           17
BeautifulPeople         362.49   1100089           20
     CyberServe         344.52   1107034           19
    MeetMindful         295.35   1422717           16


## 🧹 6. Nettoyage final

In [12]:
# Supprimer les colonnes inutiles pour l'analyse
cols_to_drop = ['LogoPath', 'DataClasses', 'Description']
cols_to_drop_present = [c for c in cols_to_drop if c in df_clean.columns]

df_clean = df_clean.drop(columns=cols_to_drop_present)
print(f"🗑️  Colonnes supprimées : {cols_to_drop_present}")

# Renommer DataClasses_parsed en DataClasses
df_clean = df_clean.rename(columns={'DataClasses_parsed': 'DataClasses'})

# Filtrer les fuites sans date valide
before = len(df_clean)
df_clean = df_clean.dropna(subset=['BreachDate'])
after = len(df_clean)
print(f"📅 Fuites sans date valide supprimées : {before - after}")

print(f"\n✅ Dataset propre : {df_clean.shape[0]} lignes x {df_clean.shape[1]} colonnes")

🗑️  Colonnes supprimées : ['LogoPath', 'DataClasses', 'Description']
📅 Fuites sans date valide supprimées : 0

✅ Dataset propre : 974 lignes x 27 colonnes


## 🔍 7. Aperçu du dataset final

In [13]:
print("📋 Colonnes du dataset propre :")
for i, col in enumerate(df_clean.columns, 1):
    print(f"  {i:02d}. {col:25s} | {str(df_clean[col].dtype):10s} | Nulls: {df_clean[col].isnull().sum()}")

📋 Colonnes du dataset propre :
  01. Name                      | object     | Nulls: 0
  02. Title                     | object     | Nulls: 0
  03. Domain                    | object     | Nulls: 52
  04. BreachDate                | datetime64[ns] | Nulls: 0
  05. AddedDate                 | datetime64[ns, UTC] | Nulls: 0
  06. ModifiedDate              | datetime64[ns, UTC] | Nulls: 0
  07. PwnCount                  | int64      | Nulls: 0
  08. Attribution               | object     | Nulls: 856
  09. DisclosureUrl             | object     | Nulls: 950
  10. IsVerified                | bool       | Nulls: 0
  11. IsFabricated              | bool       | Nulls: 0
  12. IsSensitive               | bool       | Nulls: 0
  13. IsRetired                 | bool       | Nulls: 0
  14. IsSpamList                | bool       | Nulls: 0
  15. IsMalware                 | bool       | Nulls: 0
  16. IsSubscriptionFree        | bool       | Nulls: 0
  17. IsStealerLog              | bool       |

In [14]:
# Aperçu des 5 premières lignes colonnes clés
cols_preview = ['Name', 'Year', 'PwnCount', 'BreachSize', 'Sector', 'NbDataTypes', 'SeverityScore', 'IsVerified']
cols_preview_present = [c for c in cols_preview if c in df_clean.columns]
df_clean[cols_preview_present].head(10)

,Name,Year,PwnCount,BreachSize,Sector,NbDataTypes,SeverityScore,IsVerified
0,000webhost,2015,14936670,Massive (>10M),Autre / Inconnu,4,43.05,True
1,123RF,2020,8661578,Grande (1M–10M),Autre / Inconnu,7,72.84,True
2,126,2012,6414191,Grande (1M–10M),Autre / Inconnu,2,13.61,False
3,17Media,2016,4009640,Grande (1M–10M),Autre / Inconnu,5,49.52,True
4,17173,2011,7485802,Grande (1M–10M),Autre / Inconnu,3,20.62,False
5,1win,2024,96166543,Massive (>10M),Autre / Inconnu,6,71.85,True
6,2844Breaches,2018,80115532,Massive (>10M),Autre / Inconnu,2,15.81,False
7,2fast4u,2017,17706,Petite (<100K),Autre / Inconnu,3,19.12,True
8,500px,2018,14867999,Massive (>10M),Autre / Inconnu,7,75.31,True
9,7k7k,2011,9121434,Grande (1M–10M),Autre / Inconnu,3,20.88,False


## 💾 8. Export — Dataset propre + Dataset explosé (DataClasses)

In [15]:
# ── Export 1 : Dataset principal propre ───────────────────
# Convertir la liste DataClasses en string pour le CSV
df_export = df_clean.copy()
df_export['DataClasses'] = df_export['DataClasses'].apply(lambda x: ', '.join(x) if isinstance(x, list) else '')

path_clean = '../data/breaches_clean.csv'
df_export.to_csv(path_clean, index=False, encoding='utf-8')
print(f"💾 Export 1 : {path_clean}")
print(f"   {df_export.shape[0]} lignes x {df_export.shape[1]} colonnes | {os.path.getsize(path_clean)/1024:.1f} KB")

💾 Export 1 : ../data/breaches_clean.csv
   974 lignes x 27 colonnes | 287.1 KB


In [16]:
# ── Export 2 : Dataset explosé (1 ligne par DataClass) ────
df_exploded = df_clean.explode('DataClasses').rename(columns={'DataClasses': 'DataClass'})
df_exploded = df_exploded[df_exploded['DataClass'].notna() & (df_exploded['DataClass'] != '')]

path_exploded = '../data/breaches_exploded.csv'
df_exploded.to_csv(path_exploded, index=False, encoding='utf-8')
print(f"💾 Export 2 : {path_exploded}")
print(f"   {df_exploded.shape[0]} lignes x {df_exploded.shape[1]} colonnes | {os.path.getsize(path_exploded)/1024:.1f} KB")

print(f"\n✅ Tous les fichiers exportés avec succès dans ../data/")

💾 Export 2 : ../data/breaches_exploded.csv
   5105 lignes x 27 colonnes | 1190.5 KB

✅ Tous les fichiers exportés avec succès dans ../data/


---

## ✅ Résumé du Notebook 02

| Étape | Action | Statut |
|---|---|---|
| Audit qualité | Vérification nulls et doublons | ✅ |
| Dates | Parsing BreachDate, AddedDate, ModifiedDate | ✅ |
| Feature Engineering | Year, Month, Quarter, Decade | ✅ |
| Feature Engineering | BreachSize (catégorie taille) | ✅ |
| Feature Engineering | Sector (secteur d'activité) | ✅ |
| Feature Engineering | NbDataTypes (nb types exposés) | ✅ |
| Feature Engineering | SeverityScore (score de gravité) | ✅ |
| Export | `breaches_clean.csv` | ✅ |
| Export | `breaches_exploded.csv` (1 ligne/DataClass) | ✅ |

➡️ **Prochaine étape** : `03_eda_analysis.ipynb` — Analyse exploratoire complète